# Mening ovozim — Telegram bot (Google Colab)

Matnni ravon o'qiydi (Navoiy TTS) va mening ovozimga aylantiradi (RVC), natijani Telegram'da ovozli xabar qilib yuboradi.

**Ishga tushirish:**
1. Menyu: **Runtime → Change runtime type → T4 GPU → Save**.
2. Chap paneldagi 🔑 **Secrets** bo'limiga `TELEGRAM_TOKEN` nomi bilan @BotFather bergan tokenni qo'shing va "Notebook access" ni yoqing.
3. **Runtime → Run all**. Birinchi ishga tushish ~10 daqiqa (modellar yuklanadi).
4. Telegram'da botingizga `/start` yozing — birinchi yozgan odam bot egasi bo'ladi, boshqalarga bot javob bermaydi.

Bot shu sahifa ochiq va ishlab turganda ishlaydi.

In [ ]:
# 1) Kutubxonalar va kod
!nvidia-smi -L || echo "GPU yo'q: Runtime -> Change runtime type -> T4 GPU"
!pip -q install "transformers==4.51.3" "numpy<2" "setuptools<81" conformer==0.3.2 diffusers==0.29.0 hydra-core==1.3.2 HyperPyYAML==1.2.3 inflect lightning omegaconf openai-whisper x-transformers modelscope wetext onnxruntime librosa soundfile pyarrow rich pyworld einops faiss-cpu torchcrepe torchfcpe stftpitchshift soxr noisereduce webrtcvad-wheels pyTelegramBotAPI
!rm -rf /content/Ummon && git clone -q --depth 1 -b claude/voice-lora-tts https://github.com/Shoyadbeku/Ummon.git /content/Ummon
!mkdir -p /content/app && cp /content/Ummon/voice/space/app.py /content/app/ && cp /content/Ummon/voice/trained/rvc_myvoice_30e.pth /content/app/
print("tayyor")

In [ ]:
# 2) Modellarni yuklash (birinchi marta ~10 daqiqa)
import sys, os
sys.path.insert(0, "/content/app")
os.chdir("/content/app")
import app as voice
print("Modellar tayyor")

In [ ]:
# 3) Telegram bot
import subprocess, tempfile, threading, telebot
try:
    from google.colab import userdata
    TOKEN = userdata.get("TELEGRAM_TOKEN")
except Exception:
    import getpass
    TOKEN = getpass.getpass("Telegram bot token: ")
bot = telebot.TeleBot(TOKEN)
owner = {"id": None}
busy = threading.Lock()

@bot.message_handler(commands=["start"])
def start(m):
    if owner["id"] is None:
        owner["id"] = m.from_user.id
        print("Bot egasi:", m.from_user.id)
    if m.from_user.id != owner["id"]:
        return
    bot.reply_to(m, "Salom! O'zbekcha matn yuboring, men uni sizning ovozingizda o'qib beraman.")

@bot.message_handler(content_types=["text"])
def speak(m):
    if m.from_user.id != owner["id"]:
        return
    if len(m.text) > 3000:
        bot.reply_to(m, "Matn juda uzun, 3000 belgigacha yuboring."); return
    note = bot.reply_to(m, "Ovoz yaratilmoqda…")
    with busy:
        try:
            wav = voice.synthesize(m.text)
            ogg = tempfile.mktemp(suffix=".ogg")
            subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", wav, "-c:a", "libopus", "-b:a", "48k", ogg], check=True)
            with open(ogg, "rb") as f:
                bot.send_voice(m.chat.id, f, reply_to_message_id=m.message_id)
            bot.delete_message(m.chat.id, note.message_id)
        except Exception as e:
            bot.edit_message_text(f"Xato: {e}", m.chat.id, note.message_id)

print("Bot ishlayapti. Telegram'da /start yozing. To'xtatish uchun shu katakni to'xtating.")
bot.infinity_polling(skip_pending=True)